# IOAI — 2025 National Human Vs Ai (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-human-vs-ai.zip', '/tmp/hva.zip')
    with zipfile.ZipFile('/tmp/hva.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Human vs AI — 사람 vs AI 글 분류 (베이스라인)
짧은 글이 **사람(0)** 인지 **AI 생성(1)** 인지 분류합니다. 채점 = held-out(`ID%5==0`) **accuracy**(높을수록 좋음).

베이스라인은 **단어 단위 TF-IDF + 로지스틱 회귀**. (개선: 글자 단위 `char_wb` TF-IDF + LinearSVC → 모범답안)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('data/train.csv'); df['label'] = df['label'].astype(float).astype(int)
df['text'] = df['text'].fillna('')
is_val = df['ID'] % 5 == 0
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)
print('train', len(tr), 'val', len(va), '| AI 비율', round(df['label'].mean(),3))

In [ ]:
vec = TfidfVectorizer(max_features=20000)        # 단어 단위(기본)
Xtr = vec.fit_transform(tr['text']); Xva = vec.transform(va['text'])
clf = LogisticRegression(max_iter=1000).fit(Xtr, tr['label'])
pred = clf.predict(Xva)
print('held-out accuracy:', round(accuracy_score(va['label'], pred),4),
      '| macro_f1:', round(f1_score(va['label'], pred, average='macro'),4))
pd.DataFrame({'ID': va['ID'], 'label': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)